In [ ]:
# INFUSIONMEDS.ipynb

# ======================
# Step 1. Import libraries
# ======================
import pandas as pd
import os

# 文件路径
base_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_exports"
output_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"
os.makedirs(output_path, exist_ok=True)

emar_file = os.path.join(base_path, "hosp_emar.csv")
emar_detail_file = os.path.join(base_path, "hosp_emar_detail.csv")

# ======================
# Step 2. 初始化
# ======================
chunk_size = 1_000_000   # emar 分块大小
emar_detail_chunk_size = 1_000_000   # emar_detail 分块大小

final_parts = []

# ======================
# Step 3. 分块处理 eMAR
# ======================
for i, emar_chunk in enumerate(pd.read_csv(emar_file, chunksize=chunk_size)):

    print(f"Processing EMAR chunk {i+1} ...")

    # 只保留必要字段
    emar_chunk = emar_chunk[
        ["subject_id", "hadm_id", "emar_id", "pharmacy_id", "charttime", "medication"]
    ]

    # 当前 chunk 的 keys
    keys = list(zip(emar_chunk["emar_id"], emar_chunk["pharmacy_id"], emar_chunk["subject_id"]))
    keys_df = pd.DataFrame(keys, columns=["emar_id", "pharmacy_id", "subject_id"]).drop_duplicates()

    # ======================
    # Step 4. 分块处理 eMAR_detail
    # ======================
    detail_parts = []
    for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):

        emar_detail_chunk = emar_detail_chunk[
            ["subject_id", "emar_id", "pharmacy_id", "administration_type",
             "dose_given", "dose_given_unit", "infusion_rate", "infusion_rate_unit", "route"]
        ]

        # 用 merge 筛选只和当前 emar_chunk 匹配的 detail
        emar_detail_chunk = emar_detail_chunk.merge(
            keys_df,
            on=["emar_id", "pharmacy_id", "subject_id"],
            how="inner"
        )

        if not emar_detail_chunk.empty:
            detail_parts.append(emar_detail_chunk)

    if detail_parts:
        emar_detail_all = pd.concat(detail_parts, ignore_index=True)
    else:
        emar_detail_all = pd.DataFrame(columns=[
            "subject_id", "emar_id", "pharmacy_id", "administration_type",
            "dose_given", "dose_given_unit", "infusion_rate", "infusion_rate_unit", "route"
        ])

    # ======================
    # Step 5. 合并 eMAR + eMAR_detail
    # ======================
    merged = emar_chunk.merge(
        emar_detail_all,
        on=["subject_id", "emar_id", "pharmacy_id"],
        how="left"
    )

    # ======================
    # Step 6. 构建最终 INFUSIONMEDS 表
    # ======================
    merged_final = pd.DataFrame({
        "csn": merged["hadm_id"],
        "pat_id": merged["subject_id"],
        "med_name": merged["medication"],
        "administration_type": merged["administration_type"],
        "dose_given": merged["dose_given"],
        "dose_unit": merged["dose_given_unit"],
        "infusion_rate": merged["infusion_rate"],
        "infusion_rate_unit": merged["infusion_rate_unit"],
        "route": merged["route"],
        "charttime": merged["charttime"]
    })

    final_parts.append(merged_final)

# ======================
# Step 7. 合并并保存
# ======================
infusionmeds_final = pd.concat(final_parts, ignore_index=True)

out_file = os.path.join(output_path, "INFUSIONMEDS.csv")
infusionmeds_final.to_csv(out_file, index=False)

print(f"✅ INFUSIONMEDS saved to {out_file}, shape={infusionmeds_final.shape}")


In [1]:
# INFUSIONMEDS.ipynb

# ======================
# Step 1. Import libraries
# ======================
import pandas as pd
import os

# 文件路径
base_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_exports"
output_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"
os.makedirs(output_path, exist_ok=True)

emar_file = os.path.join(base_path, "hosp_emar.csv")
emar_detail_file = os.path.join(base_path, "hosp_emar_detail.csv")

# ======================
# Step 2. 初始化
# ======================
chunk_size = 1_000_000         # emar 分块大小
emar_detail_chunk_size = 1_000_000   # emar_detail 分块大小

final_parts = []

# ======================
# Step 3. 分块处理 eMAR
# ======================
for i, emar_chunk in enumerate(pd.read_csv(emar_file, chunksize=chunk_size)):

    print(f"Processing EMAR chunk {i+1} ...")

    # 只保留必要字段
    emar_chunk = emar_chunk[
        ["subject_id", "hadm_id", "emar_id", "pharmacy_id", "charttime"]
    ]

    # 当前 chunk 的 keys
    keys_df = emar_chunk[["emar_id", "pharmacy_id", "subject_id"]].drop_duplicates()

    # ======================
    # Step 4. 分块处理 eMAR_detail
    # ======================
    detail_parts = []
    for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):

        emar_detail_chunk = emar_detail_chunk[
            ["subject_id", "emar_id", "pharmacy_id",
             "dose_given", "dose_given_unit", "route"]
        ]

        # 只保留与 emar_chunk 匹配的行
        emar_detail_chunk = emar_detail_chunk.merge(
            keys_df,
            on=["emar_id", "pharmacy_id", "subject_id"],
            how="inner"
        )

        if not emar_detail_chunk.empty:
            detail_parts.append(emar_detail_chunk)

    if detail_parts:
        emar_detail_all = pd.concat(detail_parts, ignore_index=True)
    else:
        emar_detail_all = pd.DataFrame(columns=[
            "subject_id", "emar_id", "pharmacy_id",
            "dose_given", "dose_given_unit", "route"
        ])

    # ======================
    # Step 5. 合并 eMAR + eMAR_detail
    # ======================
    merged = emar_chunk.merge(
        emar_detail_all,
        on=["subject_id", "emar_id", "pharmacy_id"],
        how="left"
    )

    # ======================
    # Step 6. 构建最终 INFUSIONMEDS 表
    # ======================
    merged_final = pd.DataFrame({
        "csn": merged["hadm_id"],
        "pat_id": merged["subject_id"],
        "medication_id": merged["pharmacy_id"],
        "med_order_time": "NOT AVAILABLE",
        "med_action_time": merged["charttime"],
        "med_start": "NOT AVAILABLE",
        "med_stop": "NOT AVAILABLE",
        "med_order_route": merged["route"],
        "med_action_dose": merged["dose_given"],
        "med_action_dose_unit": merged["dose_given_unit"]
    })

    final_parts.append(merged_final)

# ======================
# Step 7. 合并并保存
# ======================
infusionmeds_final = pd.concat(final_parts, ignore_index=True)

out_file = os.path.join(output_path, "INFUSIONMEDS.csv")
infusionmeds_final.to_csv(out_file, index=False)

print(f"✅ INFUSIONMEDS saved to {out_file}, shape={infusionmeds_final.shape}")


Processing EMAR chunk 1 ...


/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (11,14,26,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):
/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (11,14,18,21,23,25,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):
/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (8,11,14,22,25,26,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):
/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (8,11,14,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j,

Processing EMAR chunk 2 ...


/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (11,14,26,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):
/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (11,14,18,21,23,25,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):
/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (8,11,14,22,25,26,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):
/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (8,11,14,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j,

Processing EMAR chunk 3 ...


...

Processing EMAR chunk 42 ...


/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (11,14,26,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):
/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (11,14,18,21,23,25,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):
/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (8,11,14,22,25,26,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):
/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (8,11,14,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j,

Processing EMAR chunk 43 ...


/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (11,14,26,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):
/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (11,14,18,21,23,25,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):
/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (8,11,14,22,25,26,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j, emar_detail_chunk in enumerate(pd.read_csv(emar_detail_file, chunksize=emar_detail_chunk_size)):
/tmp/ipykernel_3372318/3606234789.py:44: DtypeWarning: Columns (8,11,14,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for j,

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.